# A *quickstart* notebook for **GreekBarRetrieval**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nlpaueb/greek-bar-bench/blob/main/quickstart/quickstart_greekbarretrieval.ipynb)

**GreekBarRetrieval (GBR)** is the retrieval half of GreekBarBench. Instead of being handed the
relevant legal chapters, a system must find the applicable statutory articles itself, from a pool
of **6,308 candidates** drawn from 23 Greek legal sources.

This notebook runs two baselines end to end on the public subset (257 queries, 2015–2023):

| Baseline | Kind | Needs |
|---|---|---|
| **BM25-GreekStemmer** | sparse | CPU only |
| **EmbeddingGemma-300M** | dense | any GPU, or CPU with patience |

**No API keys and no paid calls anywhere in this notebook.** On a free Colab T4 the whole thing
runs in a few minutes.

For the answering task and the LLM-judge, see the sibling notebook
[`quickstart_greekbarbench.ipynb`](https://github.com/nlpaueb/greek-bar-bench/blob/main/quickstart/quickstart_greekbarbench.ipynb).

Paper: [GreekBarRetrieval: A Benchmark for Greek Statutory Retrieval](https://arxiv.org/abs/2608.18752) (NLLP 2026).

MIT License

Copyright (c) 2025 NLP AUEB

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and
associated documentation files (the "Software"), to deal in the Software without restriction,
including without limitation the rights to use, copy, modify, merge, publish, distribute,
sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or
substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT
NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND
NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM,
DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT
OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## Install

In [ ]:
# greek-stemmer-plus is a maintained fork of greek-stemmer; the original calls yaml.load()
# without a Loader, which is a hard TypeError on the PyYAML that Colab ships.
!pip install -q "datasets>=4.0" "rank_bm25==0.2.2" "greek-stemmer-plus==0.2.0" "sentence-transformers>=5.0"

## Parameters

`REVISION` pins the dataset to the release tag, so these numbers stay reproducible even after the
repo gains the 2024 queries next year.

In [ ]:
REPO     = "AUEB-NLP/greek-bar-bench"
REVISION = "v2.0"          # pinned release tag; use a commit SHA for absolute certainty
TOP_K    = 100             # Recall@100 is the benchmark's headline metric
DENSE_MODEL = "google/embeddinggemma-300m"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

## Load the benchmark

GBR ships in the standard MTEB/BEIR layout as three subsets of the GreekBarBench repo:

| Subset | Rows | Fields |
|---|---|---|
| `corpus`  | 6,308 | `id`, `text` |
| `queries` | 257   | `id`, `text` |
| `qrels`   | 660   | `query-id`, `corpus-id`, `score` |

In [ ]:
from datasets import load_dataset
import collections

corpus  = load_dataset(REPO, "corpus",  split="test", revision=REVISION)
queries = load_dataset(REPO, "queries", split="test", revision=REVISION)
qrels   = load_dataset(REPO, "qrels",   split="test", revision=REVISION)

# gold[query_id] -> set of relevant corpus ids
gold = collections.defaultdict(set)
for row in qrels:
    gold[row["query-id"]].add(row["corpus-id"])

doc_ids = corpus["id"]
print(f"corpus  {len(corpus):>5} articles from {len({d.split('::')[0] for d in doc_ids})} legal sources")
print(f"queries {len(queries):>5}")
print(f"qrels   {len(qrels):>5}  ({len(qrels)/len(queries):.2f} gold articles per query)")

### What one query looks like

`queries.text` is the case facts followed by the question, separated by a blank line — exactly as
fed to the retrievers in the paper. Every query `id` also joins to its full record in the
`greekbarbench` subset, so you can pull the official answer or the expert span annotations for the
same question.

In [ ]:
q = queries[0]
print("id:", q["id"], "\n")
print(q["text"][:700], "...\n")
print("gold articles:", sorted(gold[q["id"]]))
for doc_id in sorted(gold[q["id"]]):
    text = corpus[doc_ids.index(doc_id)]["text"]
    print(f"\n  [{doc_id}] {text[:220]}...")

In [ ]:
# the shared id joins retrieval back to the answering task
gbb = load_dataset(REPO, "greekbarbench", split="test", revision=REVISION)
by_id = {row["id"]: row for row in gbb}
full = by_id[q["id"]]
print("area:", full["area"], "| exam:", full["date"], "| question no.", full["number"])
print("\nofficial suggested answer:\n", full["answer"][:500], "...")

## Metrics

The benchmark reports nDCG@10, nDCG@100, Recall@10, Recall@100 and MAP@100. All relevance grades
are binary (`score` is always 1), so the implementations are short enough to read:

- **nDCG@k** — discounted gain, normalised by the best achievable ordering.
- **Recall@k** — what fraction of the gold articles made the top *k*. **Recall@100 is the headline
  metric**, because retrieved articles are meant to be passed on to a generator.
- **MAP@k** — mean average precision, rewarding gold articles ranked early.

In [ ]:
import math

def ndcg_at_k(ranked, relevant, k):
    dcg  = sum(1 / math.log2(i + 2) for i, d in enumerate(ranked[:k]) if d in relevant)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg else 0.0

def average_precision_at_k(ranked, relevant, k):
    hits, total = 0, 0.0
    for i, d in enumerate(ranked[:k]):
        if d in relevant:
            hits += 1
            total += hits / (i + 1)
    return total / min(len(relevant), k) if relevant else 0.0

def evaluate(run):
    # run: {query_id: [ranked corpus_ids]}  ->  {metric: mean score}
    per_query = collections.defaultdict(list)
    for qid, ranked in run.items():
        rel = gold[qid]
        per_query["nDCG@10"].append(ndcg_at_k(ranked, rel, 10))
        per_query["nDCG@100"].append(ndcg_at_k(ranked, rel, 100))
        per_query["Recall@10"].append(len(rel & set(ranked[:10])) / len(rel))
        per_query["Recall@100"].append(len(rel & set(ranked[:100])) / len(rel))
        per_query["MAP@100"].append(average_precision_at_k(ranked, rel, 100))
    return {m: sum(v) / len(v) for m, v in per_query.items()}

## Baseline 1 — BM25 with a Greek stemmer

Greek is heavily inflected, so exact-term matching punishes BM25 badly: *ΜΙΣΘΩΣΕΩΣ*, *ΜΙΣΘΩΣΗ* and
*ΜΙΣΘΩΣΕΙΣ* are the same concept but three different tokens. The tokenizer below is the paper's
`bm25_greekstemmer_strip_diacritics` variant: strip diacritics, uppercase, drop a small stopword
list, then stem.

Stemming is doing real work here — dropping it and matching on accent-stripped surface forms costs
about 18 Recall@100 points (0.38 → 0.20).

In [ ]:
import re, unicodedata
from greek_stemmer_plus import GreekStemmer

STEMMER = GreekStemmer()

MIN_STOPWORDS = {
    "ΚΑΙ", "Η", "Ο", "ΤΟ", "ΤΑ", "ΤΟΥ", "ΤΗΣ", "ΤΩΝ", "ΣΤΟ", "ΣΤΗ", "ΣΤΗΝ", "ΣΤΑ",
    "ΣΕ", "ΜΕ", "ΓΙΑ", "ΑΠΟ", "ΩΣ", "ΠΟΥ", "ΟΤΙ", "ΟΤΑΝ", "ΑΝ", "ΝΑ", "ΔΕΝ",
    "ΜΗ", "ΜΗΝ", "ΤΙ", "ΠΩΣ", "ΠΡΟΣ", "ΥΠΟ", "ΚΑΤΑ", "ΕΝ", "ΕΠΙ", "ΜΕΤΑ",
    "ΠΡΙΝ", "ΠΑΝΩ", "ΚΑΤΩ", "ΕΩΣ", "ΗΤΑΝ", "ΕΙΝΑΙ", "ΕΙΜΑΙ", "ΕΧΕΙ", "ΕΧΟΥΝ", "ΕΧΩ",
}

def strip_diacritics(text):
    return "".join(c for c in unicodedata.normalize("NFD", text or "")
                   if unicodedata.category(c) != "Mn")

def tokenize(text):
    tokens = []
    for word in re.findall(r"\w+", text or "", flags=re.UNICODE):
        word = strip_diacritics(word).upper().strip()
        if not word or word in MIN_STOPWORDS:
            continue
        try:
            tokens.append(STEMMER.stem(word))
        except Exception:
            tokens.append(word)          # stemmer rejects some tokens (digits, latin)
    return tokens

print(tokenize("Η καταγγελία της μισθώσεως και η αποζημίωση του μισθωτή"))

In [ ]:
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

bm25 = BM25Okapi([tokenize(t) for t in tqdm(corpus["text"], desc="tokenizing corpus")])

run_bm25 = {}
for qid, qtext in zip(tqdm(queries["id"], desc="BM25 search"), queries["text"]):
    scores = bm25.get_scores(tokenize(qtext))
    top = sorted(range(len(scores)), key=lambda i: -scores[i])[:TOP_K]
    run_bm25[qid] = [doc_ids[i] for i in top]

results_bm25 = evaluate(run_bm25)
for m, v in results_bm25.items():
    print(f"  {m:12} {v:.3f}")

## Baseline 2 — EmbeddingGemma-300M (dense)

A 300M-parameter multilingual embedding model, small enough for a free Colab GPU. Its context
window is 2048 tokens, comfortably above the longest query, so nothing is truncated.

EmbeddingGemma ships **task prompts** (`task: search result | query: ` for queries,
`title: none | text: ` for documents). Counter-intuitively they *hurt* on this benchmark — encoding
the raw text scores about 2.5 nDCG@10 points higher — so this notebook encodes without them, which
is also what the paper did. The commented lines below turn them on if you want to check.

The model is gated on the Hub — accept the Gemma licence on the
[model page](https://huggingface.co/google/embeddinggemma-300m) and log in with
`huggingface_hub.login()` first, otherwise the download 401s.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(DENSE_MODEL, device=DEVICE)

doc_emb = model.encode(corpus["text"], batch_size=32,   # add prompt_name="document" to compare
                       convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
qry_emb = model.encode(queries["text"], batch_size=16,  # add prompt_name="query" to compare
                       convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
print("documents", tuple(doc_emb.shape), "| queries", tuple(qry_emb.shape))

In [ ]:
# vectors are normalised, so a dot product is cosine similarity
top = torch.topk(qry_emb @ doc_emb.T, k=TOP_K, dim=1).indices.cpu().tolist()

run_dense = {qid: [doc_ids[i] for i in idxs] for qid, idxs in zip(queries["id"], top)}

results_dense = evaluate(run_dense)
for m, v in results_dense.items():
    print(f"  {m:12} {v:.3f}")

## Results

In [ ]:
import pandas as pd

METRICS = ["nDCG@10", "nDCG@100", "Recall@10", "Recall@100", "MAP@100"]
table = pd.DataFrame(
    [[results_bm25[m] for m in METRICS], [results_dense[m] for m in METRICS]],
    index=["BM25-GreekStemmer", "EmbeddingGemma-300M"], columns=METRICS,
).round(3)
table

### Reading the numbers

Expect roughly **BM25 ≈ 0.38 Recall@100** and **EmbeddingGemma ≈ 0.39**. Two caveats when comparing
to the paper:

1. **This is the public subset.** The paper reports over all 283 queries; the 257 here exclude the
   2024 exams, which are held out. Absolute numbers shift slightly.
2. **These are the small baselines.** The paper's strongest systems are much better: Gemini-001 at
   0.77 Recall@100, and a single LLM query-reformulation pass lifts BM25 from 0.36 to 0.60.

The gap between roughly a third of the gold articles retrieved and what a citation-grounded legal
answer actually needs is the point of the benchmark.

### Where to go next

- **Query reformulation** is the cheapest large win in the paper: ask an LLM to restate the case in
  statutory terminology, and retrieve with that. It helps sparse retrieval most, because it supplies
  the legal vocabulary BM25 cannot infer from a description of everyday facts.
- **Bigger dense models** — Qwen3-Embedding-8B reaches 0.67 Recall@100 vanilla, 0.73 with
  reformulation.
- **Error analysis** — `run_bm25` and `run_dense` hold full ranked lists, so you can inspect which
  legal areas and which codes fail. Join on `id` to `greekbarbench` for the area label.

The per-area cell below is worth a look: difficulty is very uneven, and the two baselines fail in
different places. Recall@100 by area, BM25 vs dense:

| area | BM25 | EmbeddingGemma |
|---|---|---|
| lawyers | 0.66 | 0.69 |
| commercial | **0.65** | 0.49 |
| public | 0.26 | 0.29 |
| civil | 0.25 | 0.25 |
| criminal | 0.18 | **0.34** |

Commercial law favours BM25 — those questions turn on named statutes (Ν 4072/2012, Ν 5960/1933)
whose numbers appear verbatim in the query. Criminal law is nearly twice as good dense, where the
facts describe conduct that has to be matched to an abstract offence. Civil law is the hardest for
both, and it is also the largest slice of the benchmark: its gold articles are spread across ΑΚ and
ΚΠολΔ, the two biggest codes in the corpus.

In [ ]:
# per-area Recall@100, a quick way to see where retrieval breaks down
import pandas as pd

rows = []
for qid, ranked in run_bm25.items():
    rel = gold[qid]
    rows.append({"area": by_id[qid]["area"],
                 "BM25": len(rel & set(ranked[:100])) / len(rel),
                 "Dense": len(rel & set(run_dense[qid][:100])) / len(rel)})
pd.DataFrame(rows).groupby("area").agg(["mean", "count"]).round(3)